# TUTORIAL 03: ALGEBRAIC SURGERY

We now reach the absolute crown jewel of the `pysurgery` library: **Surgery Theory**.

Mathematical surgery is exactly what it sounds like. We take a geometric scalpel, cut a piece out of a manifold, and glue a different piece back into the hole. Our ultimate goal is usually to "kill" a homology class (to destroy a hole), simplifying the manifold until it becomes a perfect sphere.

In 4 dimensions, to kill a 2-dimensional hole $x$ (an embedded sphere $S^2$), we cut out its "tubular neighborhood". This neighborhood looks like $S^2 \times D^2$. The boundary of the hole we just made is $S^2 \times S^1$. We then glue in the space $D^3 \times S^1$, which has the exact same boundary! The 2-sphere is now filled in by a 3-disk. The hole is dead.

### The Obstruction: Why we can't always cut
We can only cut out $x$ if its normal bundle is trivial (a property called **framing**). 
Algebraically, this means $x$ must be **isotropic**: it must not intersect itself. So, we strictly require $Q(x, x) = 0$. If $Q(x,x) \neq 0$, the tubular neighborhood twists like a Möbius strip, and we cannot glue the replacement part in.

Furthermore, $x$ must be **primitive** (it cannot be a multiple of another class, like $2y$, otherwise the resulting space would have severe singularities).

In [1]:
import numpy as np
from pysurgery.core.intersection_forms import IntersectionForm
from pysurgery.core.exceptions import IsotropicError
print("Scalpel ready.")

/home/gabriel/Desktop/SurgeryTheory/pysurgery/core/complexes.py:69: SyntaxWarning: invalid escape sequence '\c'
  """


Scalpel ready.


## 1. Preparing the Patient: $S^2 \times S^2$

Let's try to perform surgery on $S^2 \times S^2$ to turn it into a 4-Sphere $S^4$.
As we saw in the previous tutorial, its intersection form is the hyperbolic plane $H$.

In [2]:
H = np.array([[0, 1], [1, 0]])
s2s2 = IntersectionForm(matrix=H, dimension=4)
print(f"Pre-surgery Rank: {s2s2.rank()}")
print(f"Pre-surgery Signature: {s2s2.signature()}")

Pre-surgery Rank: 2
Pre-surgery Signature: 0


## 2. Finding the Isotropic Class

We need to find a vector $x$ such that $x^T H x = 0$.
Let's choose the basis vector $x = [1, 0]^T$. Geometrically, this represents the first $S^2$ factor in $S^2 \times S^2$.

In [3]:
x_good = np.array([1, 0])
self_int = np.dot(x_good, s2s2.matrix @ x_good)

print(f"Let x = {x_good}. Self-intersection Q(x,x) = {self_int}.")
print("It is Isotropic! Because the self-intersection is exactly 0, the normal bundle is trivial.")
print("We can legally proceed with surgery.")

Let x = [1 0]. Self-intersection Q(x,x) = 0.
It is Isotropic! Because the self-intersection is exactly 0, the normal bundle is trivial.
We can legally proceed with surgery.


## 3. Performing the Algebraic Surgery

When we call `perform_algebraic_surgery(x)`, `pysurgery` does not just guess. It executes several highly complex, rigorous mathematical algorithms to compute the exact topology of the new manifold:

1. **Finding the Dual (Extended Euclidean Algorithm)**: We can't just delete $x$. We must delete the entire hyperbolic plane. `pysurgery` uses the Euclidean algorithm to solve a Diophantine equation, finding a dual class $y$ such that $Q(x,y) = 1$.
2. **Orthogonal Projection**: It mathematically "cuts out" the plane spanned by $x$ and $y$. It projects the entire intersection lattice onto the orthogonal complement $H^\perp = \{x,y\}^\perp$ using the formula: $P(v) = v - [Q(v,y) - Q(y,y)Q(v,x)]x - Q(v,x)y$.
3. **Lattice Reduction (Hermite Normal Form)**: We are working over integers $\mathbb{Z}$, not real numbers. We cannot just use Gram-Schmidt. `pysurgery` uses SymPy's **Hermite Normal Form (HNF)** to extract a perfect, new integer $\mathbb{Z}$-basis for this restricted lattice.

The returned object is the new Intersection Form.

In [4]:
print("Executing algebraic surgery...")
surgered_manifold = s2s2.perform_algebraic_surgery(x_good)

print("\n--- RESULTING MANIFOLD ---")
print(f"Post-Surgery Matrix:\n{surgered_manifold.matrix}")
print(f"Rank: {surgered_manifold.rank()}")

print("\nExplanation:")
print("Because the rank is 0, the matrix is completely empty. There are no 2-dimensional holes left.")
print("We have successfully surgered S^2 x S^2 into a homology S^4 sphere!")

Executing algebraic surgery...

--- RESULTING MANIFOLD ---
Post-Surgery Matrix:
[]
Rank: 0

Explanation:
Because the rank is 0, the matrix is completely empty. There are no 2-dimensional holes left.
We have successfully surgered S^2 x S^2 into a homology S^4 sphere!


## 4. The Obstruction: Why we can't surger everything

Surgery seems like magic. Why don't we just use it to turn every manifold into a sphere?

Let's try to turn $\mathbb{CP}^2$ into a sphere. Its matrix is just `[1]`. 
Any vector $v = [a]$ will have $Q(v,v) = a \cdot 1 \cdot a = a^2$.
For $a^2$ to be 0, $a$ must be 0. But the zero vector isn't a hole!

Because the intersection form is **positive-definite**, **no non-trivial isotropic vectors exist**. 
If you try to cut it, `pysurgery` will detect this impossible geometry and raise a rigorous topological exception.

In [5]:
cp2 = IntersectionForm(matrix=np.array([[1]]), dimension=4)
x_bad = np.array([1])

print("Attempting to surger CP^2...")
try:
    cp2.perform_algebraic_surgery(x_bad)
except IsotropicError as e:
    print(f"\n>>> TOPOLOGICAL ERROR CAUGHT: {e}")
    print("This is the mathematical proof that CP^2 cannot be simplified into a sphere via surgery.")
    print("The signature of the form is a permanent topological obstruction.")

Attempting to surger CP^2...

>>> TOPOLOGICAL ERROR CAUGHT: Class x is not isotropic (self-intersection is not 0).
This is the mathematical proof that CP^2 cannot be simplified into a sphere via surgery.
The signature of the form is a permanent topological obstruction.
